# Autoimmune Disease Feature Extraction Pipeline

Multi-source feature extraction from GWAS Catalog, IEDB, and HPO for autoimmune disease modeling.

**Data sources:**
- GWAS Catalog: genetic association features
- IEDB: immunoepitope (self-antigen + cytokine) features
- HPO: organ system involvement and phenotype features

## 0. System Diagnostics

In [ ]:
import torch
import torch_geometric
import psutil
import GPUtil

print("=" * 60)
print("System Memory Diagnostics")
print("=" * 60)

cpu_mem = psutil.virtual_memory()
print(f"\nCPU Memory:")
print(f"  Total:     {cpu_mem.total / 1e9:.2f} GB")
print(f"  Available: {cpu_mem.available / 1e9:.2f} GB")
print(f"  Used:      {cpu_mem.percent}%")

if torch.cuda.is_available():
    print(f"\nGPU Memory:")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\n  GPU {i}: {props.name}")
        print(f"    Total:     {props.total_memory / 1e9:.2f} GB")
        print(f"    Allocated: {torch.cuda.memory_allocated(i) / 1e9:.2f} GB")
        print(f"    Reserved:  {torch.cuda.memory_reserved(i) / 1e9:.2f} GB")
        print(f"    Free:      {(props.total_memory - torch.cuda.memory_reserved(i)) / 1e9:.2f} GB")
else:
    print("\nNo GPU available.")

print("\n" + "=" * 60)

## 1. GWAS Feature Extraction — Simple Version

Lightweight function for quick prototyping.

In [ ]:
import pandas as pd
import numpy as np
import re


def extract_gwas_features(mondo_input_path, gwas_data_path, output_path):
    """Extract GWAS genetic features for each MONDO disease.

    Matches MONDO disease labels against GWAS Catalog DISEASE/TRAIT field
    and computes per-disease genetic summary statistics.

    Args:
        mondo_input_path: Path to CSV with columns ['formatted_id', 'Preferred Label'].
        gwas_data_path:   Path to GWAS Catalog TSV (full associations file).
        output_path:      Path for the output CSV.

    Returns:
        DataFrame with per-disease genetic features.
    """
    print("Loading MONDO list and GWAS data...")
    mondo_list = pd.read_csv(mondo_input_path)

    use_cols = ['DISEASE/TRAIT', 'PVALUE_MLOG', 'OR or BETA', 'CHR_ID', 'CHR_POS', 'MAPPED_GENE']
    gwas_df = pd.read_csv(gwas_data_path, sep='\t', usecols=use_cols, low_memory=False)

    # Coerce numeric columns
    gwas_df['PVALUE_MLOG'] = pd.to_numeric(gwas_df['PVALUE_MLOG'], errors='coerce').fillna(0)
    gwas_df['OR or BETA'] = pd.to_numeric(gwas_df['OR or BETA'], errors='coerce').fillna(1.0)
    # OR/BETA: Odds Ratio or effect size from GWAS association

    results = []
    print("Matching MONDO IDs to GWAS associations...")

    for _, row in mondo_list.iterrows():
        mondo_id = row['formatted_id']
        label = row['Preferred Label']

        # Fuzzy match on DISEASE/TRAIT column
        mask = gwas_df['DISEASE/TRAIT'].str.contains(re.escape(label), case=False, na=False)
        hits = gwas_df[mask]

        if hits.empty:
            results.append({
                'mondo_id': mondo_id,
                'gwas_snp_count': 0,
                'max_neg_log_p': 0,
                'median_or_value': 1.0,
                'hla_hit_ratio': 0,
                'affected_gene_count': 0,
                'genetic_intensity_score': 0,
            })
            continue

        snp_count = len(hits)
        max_p = hits['PVALUE_MLOG'].max()
        median_or = hits['OR or BETA'].median()

        # HLA region: chr6 25–35 Mb
        hla_hits = hits[
            (hits['CHR_ID'].astype(str) == '6') &
            (pd.to_numeric(hits['CHR_POS'], errors='coerce').between(25_000_000, 35_000_000))
        ]
        hla_ratio = len(hla_hits) / snp_count

        gene_count = hits['MAPPED_GENE'].dropna().nunique()
        intensity = (snp_count * hits['PVALUE_MLOG'].mean()) / 100

        results.append({
            'mondo_id': mondo_id,
            'gwas_snp_count': snp_count,
            'max_neg_log_p': max_p,
            'median_or_value': median_or,
            'hla_hit_ratio': hla_ratio,
            'affected_gene_count': gene_count,
            'genetic_intensity_score': intensity,
        })

    final_gwas_df = pd.DataFrame(results)
    final_gwas_df.to_csv(output_path, index=False)
    print(f"Done. Results saved to: {output_path}")
    return final_gwas_df


# --- Run ---
mondo_input = 'output_data/autoimmune_subclasses.csv'
gwas_data   = 'original_data/gwas-catalog-download-associations-alt-full.tsv'
output_file = 'output_data/gwas_genetic_features.csv'

gwas_features = extract_gwas_features(mondo_input, gwas_data, output_file)

## 2. GWAS Feature Extraction — Full Version

Production-grade extraction with dual matching strategy (URI + name), HLA tagging,
significance filtering (p < 5e-8), and rich derived features.

In [ ]:
#!/usr/bin/env python3
"""GWAS Catalog genetic feature extraction for autoimmune diseases."""

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Configuration ────────────────────────────────────────────────────────────
INPUT_DIR  = Path("original_data")
OUTPUT_DIR = Path("output_data")
MONDO_FILE  = OUTPUT_DIR / "autoimmune_subclasses.csv"
GWAS_FILE   = INPUT_DIR  / "gwas_catalog.tsv"  # adjust filename as needed
OUTPUT_FILE = OUTPUT_DIR / "gwas_genetic_features.csv"

# HLA / MHC region on chromosome 6
HLA_CHROMOSOME = "6"
HLA_START = 28_000_000
HLA_END   = 34_000_000


# ── Data loading ─────────────────────────────────────────────────────────────

def load_mondo_list(file_path):
    """Load MONDO disease list."""
    print(f"Loading MONDO list: {file_path}")
    df = pd.read_csv(file_path)
    print(f"  {len(df)} disease entries")
    return df


def load_gwas_data(file_path):
    """Load GWAS Catalog TSV."""
    print(f"\nLoading GWAS data: {file_path}")
    df = pd.read_csv(file_path, sep='\t', low_memory=False)
    print(f"  {len(df)} association records")
    return df


# ── Preprocessing ────────────────────────────────────────────────────────────

def preprocess_gwas_data(df):
    """Coerce types, tag HLA region, and filter to genome-wide significance."""
    print("\nPreprocessing GWAS data...")

    df['P-VALUE'] = pd.to_numeric(df['P-VALUE'], errors='coerce')
    df['neg_log_p'] = -np.log10(df['P-VALUE'].replace(0, np.nan))
    df['OR or BETA'] = pd.to_numeric(df['OR or BETA'], errors='coerce')

    df['CHR_ID'] = df['CHR_ID'].astype(str)
    df['CHR_POS'] = pd.to_numeric(df['CHR_POS'], errors='coerce')

    df['is_hla'] = (
        (df['CHR_ID'] == HLA_CHROMOSOME) &
        df['CHR_POS'].between(HLA_START, HLA_END)
    ).astype(int)

    # Retain genome-wide significant hits only (standard GWAS threshold)
    df = df[df['P-VALUE'] < 5e-8].copy()
    print(f"  Significant associations retained: {len(df)} (p < 5e-8)")
    return df


# ── Matching ─────────────────────────────────────────────────────────────────

def match_mondo_to_gwas(mondo_df, gwas_df):
    """Match MONDO diseases to GWAS records via URI and disease name.

    Strategy 1: direct URI match in MAPPED_TRAIT_URI.
    Strategy 2: keyword match on DISEASE/TRAIT (conservative, long words only).
    """
    print("\nMatching MONDO to GWAS...")
    matched_records = []

    stop_words = {'disease', 'syndrome', 'disorder'}

    for _, row in mondo_df.iterrows():
        mondo_id = row['formatted_id']
        disease_name = row['Preferred Label'].lower()

        # Strategy 1: URI match
        uri_matches = gwas_df[
            gwas_df['MAPPED_TRAIT_URI'].fillna('').str.contains(
                mondo_id, case=False, na=False, regex=False
            )
        ].copy()

        # Strategy 2: keyword name match
        keywords = [w for w in disease_name.split() if len(w) > 4 and w not in stop_words]
        name_matches = pd.DataFrame()
        if keywords:
            trait_col = gwas_df['DISEASE/TRAIT'].fillna('').str.lower()
            mask = pd.Series(False, index=gwas_df.index)
            for kw in keywords:
                try:
                    mask |= trait_col.str.contains(kw, case=False, na=False, regex=False)
                except Exception:
                    continue
            name_matches = gwas_df[mask].copy()

        combined = pd.concat([uri_matches, name_matches]).drop_duplicates()
        if len(combined) > 0:
            combined['mondo_id'] = mondo_id
            combined['disease_name'] = row['Preferred Label']
            matched_records.append(combined)
            print(f"  {mondo_id}: {len(combined)} records")
        else:
            print(f"  {mondo_id}: no match")

    if matched_records:
        result = pd.concat(matched_records, ignore_index=True)
        print(f"\nMatched {len(result)} records across {result['mondo_id'].nunique()} diseases")
        return result
    return pd.DataFrame()


# ── Feature extraction ───────────────────────────────────────────────────────

def extract_genetic_features(matched_df):
    """Aggregate per-disease genetic features from matched GWAS records."""
    print("\nExtracting genetic features...")
    if matched_df.empty:
        return pd.DataFrame()

    features = matched_df.groupby('mondo_id').agg(
        gwas_snp_count     = ('SNP_ID_CURRENT',    'nunique'),
        max_neg_log_p      = ('neg_log_p',          'max'),
        mean_neg_log_p     = ('neg_log_p',          'mean'),
        hla_hit_ratio      = ('is_hla',             'mean'),
        hla_snp_count      = ('is_hla',             'sum'),
        peak_or_value      = ('OR or BETA',         lambda x: x.abs().max()),
        mean_or_value      = ('OR or BETA',         lambda x: x.abs().mean()),
        affected_gene_count= ('REPORTED GENE(S)',   'nunique'),
        chromosome_diversity=('CHR_ID',             'nunique'),
        study_count        = ('PUBMEDID',           'nunique'),
    ).reset_index()

    disease_names = matched_df[['mondo_id', 'disease_name']].drop_duplicates()
    features = features.merge(disease_names, on='mondo_id', how='left')

    # Derived composite scores
    features['genetic_intensity_score'] = np.log1p(features['gwas_snp_count']) * features['max_neg_log_p']
    features['hla_dependency_score']    = features['hla_hit_ratio'] * np.log1p(features['hla_snp_count'])
    features['genetic_complexity']      = np.log1p(features['affected_gene_count']) * np.log1p(features['chromosome_diversity'])
    features['confidence_score']        = np.log1p(features['study_count']) * features['max_neg_log_p']

    col_order = [
        'mondo_id', 'disease_name',
        'gwas_snp_count', 'max_neg_log_p', 'mean_neg_log_p',
        'hla_hit_ratio', 'hla_snp_count', 'hla_dependency_score',
        'peak_or_value', 'mean_or_value',
        'affected_gene_count', 'chromosome_diversity',
        'study_count', 'genetic_intensity_score',
        'genetic_complexity', 'confidence_score',
    ]
    features = features[col_order]
    print(f"Done: {len(features)} diseases, {len(col_order) - 2} features")
    return features


def print_summary(features_df):
    """Print feature summary statistics."""
    print("\n" + "=" * 60)
    print("Feature Summary Statistics")
    print("=" * 60)
    numeric_cols = features_df.select_dtypes(include=[np.number]).columns
    print(features_df[numeric_cols].describe().round(3))

    # Diseases with high HLA involvement
    high_hla = features_df[features_df['hla_hit_ratio'] > 0.5].sort_values('hla_hit_ratio', ascending=False)
    print("\nHigh HLA involvement (ratio > 50%):")
    if len(high_hla) > 0:
        for _, r in high_hla.head(10).iterrows():
            print(f"  {r['disease_name']}: {r['hla_hit_ratio']:.2%} ({int(r['hla_snp_count'])} HLA SNPs)")
    else:
        print("  None")


# ── Main ─────────────────────────────────────────────────────────────────────

def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    mondo_df = load_mondo_list(MONDO_FILE)
    gwas_df  = load_gwas_data(GWAS_FILE)
    gwas_df  = preprocess_gwas_data(gwas_df)

    matched_df = match_mondo_to_gwas(mondo_df, gwas_df)

    if matched_df.empty:
        print("No matching records found.")
        return

    features_df = extract_genetic_features(matched_df)
    features_df.to_csv(OUTPUT_FILE, index=False)
    print(f"\nFeatures saved to: {OUTPUT_FILE}")

    # Save matched details for inspection
    detail_file = OUTPUT_DIR / 'gwas_matched_details.csv'
    matched_df[[
        'mondo_id', 'disease_name', 'DISEASE/TRAIT',
        'REPORTED GENE(S)', 'STRONGEST SNP-RISK ALLELE', 'P-VALUE', 'is_hla'
    ]].to_csv(detail_file, index=False)
    print(f"Match details saved to: {detail_file}")

    print_summary(features_df)


if __name__ == '__main__':
    main()

## 3. IEDB Immunoepitope Feature Extraction

Builds a one-hot feature matrix (~55-dim) from IEDB T-cell assay data:
top self-antigens (45 features) + key cytokines (10 features).

In [ ]:
#!/usr/bin/env python3
"""IEDB immunoepitope feature extraction for autoimmune diseases.

Produces:
  - iedb_onehot_features.csv     : binary self-antigen + cytokine matrix
  - iedb_statistical_features.csv: epitope count statistics
  - iedb_feature_vocabulary.csv  : feature name reference
"""

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')

# ── Configuration ────────────────────────────────────────────────────────────
INPUT_DIR  = Path("original_data")
OUTPUT_DIR = Path("output_data")
MONDO_FILE   = OUTPUT_DIR / "autoimmune_subclasses.csv"
TCELL_FILE   = INPUT_DIR  / "tcell_full_v3.csv"
ANTIGEN_FILE = INPUT_DIR  / "antigen_full_v3.csv"

OUTPUT_ONEHOT_FILE = OUTPUT_DIR / "iedb_onehot_features.csv"
OUTPUT_STAT_FILE   = OUTPUT_DIR / "iedb_statistical_features.csv"
OUTPUT_VOCAB_FILE  = OUTPUT_DIR / "iedb_feature_vocabulary.csv"

# Human species identifiers (self-antigen filter)
HUMAN_SPECIES = ["Homo sapiens", "Human", "http://purl.obolibrary.org/obo/NCBITaxon_9606"]

# Autoimmune-relevant cytokines
KEY_CYTOKINES = [
    'IL-2', 'IL-4', 'IL-6', 'IL-10', 'IL-17', 'IL-23',
    'IFN-gamma', 'IFN-alpha', 'IFN-beta', 'TNF-alpha', 'TNF-beta',
    'TGF-beta', 'GM-CSF', 'IL-1beta', 'IL-12', 'IL-21',
]

TARGET_ONEHOT_DIM = 55  # 45 self-antigens + 10 cytokines (adjust as needed)


# ── Helpers ──────────────────────────────────────────────────────────────────

def load_mondo_list(file_path):
    """Load MONDO disease list."""
    print(f"Loading MONDO list: {file_path}")
    df = pd.read_csv(file_path)
    print(f"  {len(df)} diseases")
    return df


def load_iedb_data(tcell_file, antigen_file):
    """Load IEDB T-cell assay and antigen data (skip header group row)."""
    print("\nLoading IEDB data...")
    tcell_df  = pd.read_csv(tcell_file,  skiprows=1)
    antigen_df = pd.read_csv(antigen_file, skiprows=1)
    print(f"  T-cell records:  {len(tcell_df)}")
    print(f"  Antigen records: {len(antigen_df)}")
    return tcell_df, antigen_df


def identify_self_antigens(df):
    """Flag records where the source organism is human."""
    organism_cols = [c for c in df.columns if 'Source Organism' in c or 'Organism' in c]
    is_self = pd.Series(False, index=df.index)
    for col in organism_cols:
        for term in HUMAN_SPECIES:
            is_self |= df[col].fillna('').str.contains(term, case=False, na=False, regex=False)
    df['is_self_antigen'] = is_self
    print(f"  Self-antigen records: {is_self.sum()} ({is_self.mean()*100:.1f}%)")
    return df


def match_mondo_to_iedb(mondo_df, tcell_df):
    """Match MONDO diseases to IEDB T-cell records by disease name keywords."""
    print("\nMatching MONDO to IEDB...")
    disease_col = next(
        (c for c in tcell_df.columns if 'Disease' in c and 'IRI' not in c and 'Stage' not in c),
        None
    )
    if disease_col is None:
        print("  Disease column not found.")
        return pd.DataFrame()

    stop_words = {'disease', 'syndrome', 'disorder', 'type'}
    matched_records = []

    for _, row in mondo_df.iterrows():
        mondo_id = row['formatted_id']
        disease_name = row['Preferred Label'].lower()
        keywords = [w for w in disease_name.split() if len(w) > 4 and w not in stop_words]
        if not keywords:
            continue

        disease_series = tcell_df[disease_col].fillna('').str.lower()
        mask = pd.Series(False, index=tcell_df.index)
        for kw in keywords:
            try:
                mask |= disease_series.str.contains(kw, case=False, na=False, regex=False)
            except Exception:
                continue

        matched = tcell_df[mask].copy()
        if len(matched) > 0:
            matched['mondo_id'] = mondo_id
            matched['disease_name'] = row['Preferred Label']
            matched_records.append(matched)
            print(f"  {mondo_id}: {len(matched)} records")
        else:
            print(f"  {mondo_id}: no match")

    if matched_records:
        result = pd.concat(matched_records, ignore_index=True)
        print(f"\nMatched {len(result)} records across {result['mondo_id'].nunique()} diseases")
        return result
    return pd.DataFrame()


def extract_antigen_names(df):
    """Extract and normalize antigen names from Source Molecule columns."""
    src_mol_cols = [c for c in df.columns if 'Source Molecule' in c and 'IRI' not in c]
    names = []
    for col in src_mol_cols:
        vals = df[col].fillna('').astype(str).str.strip()
        names.extend(vals[vals != ''].tolist())
    return names


def build_top_antigen_vocabulary(matched_df, top_n=45):
    """Select the most frequent self-antigens across all matched diseases."""
    print(f"\nBuilding top-{top_n} self-antigen vocabulary...")
    self_df = matched_df[matched_df['is_self_antigen']]
    counter = Counter(extract_antigen_names(self_df))
    top_antigens = [name for name, _ in counter.most_common(top_n)]
    print("  Top 5 antigens:")
    for name, cnt in counter.most_common(5):
        print(f"    {name}: {cnt}")
    return top_antigens, counter


def create_onehot_features(matched_df, antigen_vocab, cytokine_vocab):
    """Build binary feature matrix: [self-antigen presence | cytokine presence]."""
    print(f"\nBuilding one-hot feature matrix ({len(antigen_vocab)} antigens + {len(cytokine_vocab)} cytokines)...")
    mondo_ids = matched_df['mondo_id'].unique()
    feature_matrix = pd.DataFrame(0, index=mondo_ids, columns=antigen_vocab + cytokine_vocab)

    # Self-antigen features
    for mid in mondo_ids:
        disease_data = matched_df[(matched_df['mondo_id'] == mid) & matched_df['is_self_antigen']]
        for antigen in extract_antigen_names(disease_data):
            if antigen in antigen_vocab:
                feature_matrix.loc[mid, antigen] = 1

    # Cytokine features (search response/measurement columns)
    response_cols = [c for c in matched_df.columns
                     if 'Response' in c or 'measured' in c or 'cytokine' in c.lower()]
    for mid in mondo_ids:
        disease_data = matched_df[matched_df['mondo_id'] == mid]
        for col in response_cols:
            if col in disease_data.columns:
                text = ' '.join(disease_data[col].fillna('').astype(str))
                for cytokine in cytokine_vocab:
                    if cytokine.lower() in text.lower():
                        feature_matrix.loc[mid, cytokine] = 1

    feature_matrix = feature_matrix.reset_index().rename(columns={'index': 'mondo_id'})
    data = feature_matrix.iloc[:, 1:].values
    sparsity = 1 - np.count_nonzero(data) / data.size
    print(f"  Shape: {feature_matrix.shape}, sparsity: {sparsity*100:.1f}%")
    return feature_matrix


def create_statistical_features(matched_df):
    """Compute per-disease epitope count statistics."""
    print("\nComputing statistical features...")
    stats = matched_df.groupby('mondo_id').agg(
        total_epitopes     =('mondo_id',          'count'),
        self_antigen_ratio =('is_self_antigen',   'mean'),
        self_antigen_count =('is_self_antigen',   'sum'),
        unique_antigens    =('mondo_id', lambda x: len(set(extract_antigen_names(
            matched_df[matched_df['mondo_id'].isin(x)])
        ))),
    ).reset_index()
    disease_names = matched_df[['mondo_id', 'disease_name']].drop_duplicates()
    stats = stats.merge(disease_names, on='mondo_id', how='left')
    print(f"  Shape: {stats.shape}")
    return stats


def save_feature_vocabulary(antigen_vocab, cytokine_vocab, antigen_counter):
    """Save feature vocabulary for model interpretability."""
    vocab_data = (
        [{'feature_type': 'self_antigen', 'feature_name': a, 'frequency': antigen_counter.get(a, 0)}
         for a in antigen_vocab] +
        [{'feature_type': 'cytokine',     'feature_name': c, 'frequency': 0}
         for c in cytokine_vocab]
    )
    vocab_df = pd.DataFrame(vocab_data)
    vocab_df.to_csv(OUTPUT_VOCAB_FILE, index=False)
    print(f"  Vocabulary saved to: {OUTPUT_VOCAB_FILE}")
    return vocab_df


# ── Main ─────────────────────────────────────────────────────────────────────

def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    mondo_df           = load_mondo_list(MONDO_FILE)
    tcell_df, _        = load_iedb_data(TCELL_FILE, ANTIGEN_FILE)
    tcell_df           = identify_self_antigens(tcell_df)
    matched_df         = match_mondo_to_iedb(mondo_df, tcell_df)

    if matched_df.empty:
        print("No matching data found.")
        return

    n_cytokines = len(KEY_CYTOKINES)
    antigen_vocab, antigen_counter = build_top_antigen_vocabulary(
        matched_df, top_n=TARGET_ONEHOT_DIM - n_cytokines
    )
    cytokine_vocab = KEY_CYTOKINES

    onehot_features = create_onehot_features(matched_df, antigen_vocab, cytokine_vocab)
    stat_features   = create_statistical_features(matched_df)

    print("\nSaving outputs...")
    onehot_features.to_csv(OUTPUT_ONEHOT_FILE, index=False)
    stat_features.to_csv(OUTPUT_STAT_FILE,     index=False)
    save_feature_vocabulary(antigen_vocab, cytokine_vocab, antigen_counter)

    print(f"\nOne-hot features : {onehot_features.shape}")
    print(f"Statistical feats: {stat_features.shape}")
    print(f"Antigen features : {len(antigen_vocab)}")
    print(f"Cytokine features: {len(cytokine_vocab)}")


if __name__ == '__main__':
    main()

## 4. Clinical Antibody Feature Dictionary

Prior-knowledge antibody feature mapping for autoimmune diseases.
Covers SSc, myositis, vasculitis, neurology, liver, and other subtypes.

In [ ]:
EXTENDED_AB_FEATURES = {
    # Core / multi-disease
    'feat_ab_dsDNA':       ['dsDNA', 'double-stranded DNA'],
    'feat_ab_Sm':          ['Smith antigen', 'Sm antigen', 'snRNP'],
    'feat_ab_Ro_SSA':      ['Ro60', 'SSA', 'Trove2', 'Ro/SSA'],
    'feat_ab_La_SSB':      ['La/SSB', 'SSB', 'Lupus La'],
    'feat_ab_CCP':         ['CCP', 'Citrullinated', 'Filaggrin'],
    'feat_ab_RF':          ['Rheumatoid factor', 'IgG-Fc'],
    'feat_ab_Insulin':     ['Insulin', 'INS'],
    'feat_ab_GAD65':       ['GAD65', 'Glutamate decarboxylase'],
    'feat_ab_TPO':         ['Thyroid peroxidase', 'TPO'],
    'feat_ab_TG':          ['Thyroglobulin'],
    'feat_ab_MBP':         ['Myelin basic protein', 'MBP'],
    'feat_ab_Cardiolipin': ['Cardiolipin', 'Antiphospholipid', 'Beta-2-Glycoprotein'],
    'feat_ab_Histone':     ['Histone'],

    # Systemic sclerosis / scleroderma
    'feat_ab_Scl70':              ['Scl-70', 'Topoisomerase I', 'Scl70'],
    'feat_ab_Centromere':         ['Centromere', 'CENP-A', 'CENP-B', 'CENP'],
    'feat_ab_RNA_Polymerase_III': ['RNA polymerase III', 'RPC155', 'POLR3A'],

    # Myositis-specific (PM/DM)
    'feat_ab_Jo1':  ['Jo-1', 'Histidyl-tRNA synthetase', 'Jo1'],
    'feat_ab_Mi2':  ['Mi-2', 'Mi2', 'Nucleosome remodeling deacetylase'],
    'feat_ab_MDA5': ['MDA5', 'IFIH1', 'Melanoma differentiation-associated protein 5'],
    'feat_ab_TIF1g':['TIF1-gamma', 'TIF1g', 'TRIM33'],

    # Vasculitis (ANCA)
    'feat_ab_MPO': ['Myeloperoxidase', 'MPO', 'p-ANCA'],
    'feat_ab_PR3': ['Proteinase 3', 'PR3', 'c-ANCA'],
    'feat_ab_GBM': ['Glomerular basement membrane', 'GBM', 'COL4A3'],

    # Neurological autoimmune
    'feat_ab_AChR': ['Acetylcholine receptor', 'AChR'],
    'feat_ab_MuSK': ['MuSK', 'Muscle-specific kinase'],
    'feat_ab_AQP4': ['Aquaporin-4', 'AQP4', 'NMO-IgG'],
    'feat_ab_MOG':  ['Myelin oligodendrocyte glycoprotein', 'MOG'],
    'feat_ab_NMDA': ['NMDA receptor', 'NMDAR', 'GRIN1'],

    # GI / hepatic autoimmune
    'feat_ab_AMA':  ['Antimitochondrial', 'AMA-M2', 'Pyruvate dehydrogenase'],
    'feat_ab_ASCA': ['Saccharomyces cerevisiae antibody', 'ASCA'],
    'feat_ab_LKM1': ['Liver kidney microsome', 'LKM-1', 'CYP2D6'],

    # Other
    'feat_ab_U1RNP':      ['U1-RNP', 'U1RNP', 'RNP-70k'],
    'feat_ab_B2GP1':      ['Beta-2-glycoprotein I', 'B2GPI', 'APOA4'],
    'feat_ab_Fibrillarin':['Fibrillarin', 'U3-RNP'],
}

## 5. HPO Phenotype Feature Extraction

Extracts organ system involvement (11-dim one-hot) and phenotype statistics from HPO annotations.

> **Note:** Run with `--test` argument (via `sys.argv`) to process only the first 3 diseases for quick validation.

In [ ]:
#!/usr/bin/env python3
"""HPO phenotype feature extraction for autoimmune diseases.

Produces:
  - hpo_organ_features.csv    : 11-dim organ system one-hot features
  - hpo_phenotype_stats.csv   : per-disease phenotype statistics
  - mondo_to_omim_mapping.csv : MONDO <-> HPO disease ID mapping
"""

import sys
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Configuration ────────────────────────────────────────────────────────────
INPUT_DIR  = Path("original_data")
OUTPUT_DIR = Path("output_data")
MONDO_FILE = OUTPUT_DIR / "autoimmune_subclasses.csv"
HPO_FILE   = INPUT_DIR  / "phenotype.hpoa"

OUTPUT_ORGAN_FILE   = OUTPUT_DIR / "hpo_organ_features.csv"
OUTPUT_PHENOTYPE_FILE = OUTPUT_DIR / "hpo_phenotype_stats.csv"
OUTPUT_MAPPING_FILE = OUTPUT_DIR / "mondo_to_omim_mapping.csv"

# HPO organ system definitions (top-level HPO categories relevant to autoimmunity)
HPO_ORGAN_SYSTEMS = {
    'immune_system':    {'hpo_root': 'HP:0002715', 'keywords': ['immune', 'autoimmune', 'immunodeficiency', 'lymph', 'antibody']},
    'skin_connective':  {'hpo_root': 'HP:0001574', 'keywords': ['skin', 'rash', 'dermatitis', 'integument', 'connective tissue']},
    'musculoskeletal':  {'hpo_root': 'HP:0033127', 'keywords': ['joint', 'arthritis', 'bone', 'skeletal', 'muscle', 'myopathy']},
    'nervous_system':   {'hpo_root': 'HP:0000707', 'keywords': ['neurological', 'neuropathy', 'brain', 'nerve', 'seizure']},
    'renal_urinary':    {'hpo_root': 'HP:0000119', 'keywords': ['kidney', 'renal', 'nephritis', 'glomerulo', 'urinary']},
    'gastrointestinal': {'hpo_root': 'HP:0025031', 'keywords': ['gastrointestinal', 'intestinal', 'bowel', 'digestive', 'liver', 'hepatic']},
    'cardiovascular':   {'hpo_root': 'HP:0001626', 'keywords': ['cardiac', 'heart', 'vascular', 'vasculitis', 'cardiovascular']},
    'respiratory':      {'hpo_root': 'HP:0002086', 'keywords': ['pulmonary', 'lung', 'respiratory', 'pneumonitis']},
    'hematologic':      {'hpo_root': 'HP:0001871', 'keywords': ['blood', 'anemia', 'thrombocytopenia', 'leukopenia', 'hematologic']},
    'endocrine':        {'hpo_root': 'HP:0000818', 'keywords': ['thyroid', 'endocrine', 'diabetes', 'adrenal', 'hormone']},
    'ocular':           {'hpo_root': 'HP:0000478', 'keywords': ['eye', 'ocular', 'visual', 'uveitis', 'ophthalm']},
}


# ── Data loading ─────────────────────────────────────────────────────────────

def load_mondo_list(file_path):
    """Load MONDO disease list and extract numeric MONDO ID."""
    print(f"Loading MONDO list: {file_path}")
    df = pd.read_csv(file_path)
    df['mondo_number'] = df['formatted_id'].str.extract(r'MONDO:(\d+)')
    print(f"  {len(df)} diseases")
    return df


def load_hpo_annotations(file_path):
    """Load HPO HPOA annotation file (TSV with comment lines)."""
    print(f"\nLoading HPO annotations: {file_path}")
    try:
        df = pd.read_csv(file_path, sep='\t', comment='#')
        print(f"  {len(df)} records loaded")
        return df
    except Exception as e:
        print(f"  Failed to load: {e}")
        return None


def print_db_sources(hpo_df):
    """Print distribution of disease database sources (OMIM, ORPHANET, etc.)."""
    if 'database_id' not in hpo_df.columns:
        return
    db_sources = hpo_df['database_id'].str.extract(r'^([A-Z]+):')[0].value_counts()
    print("  Database sources:")
    for db, cnt in db_sources.items():
        print(f"    {db}: {cnt}")


# ── Matching ─────────────────────────────────────────────────────────────────

def build_mondo_mapping(mondo_df):
    """Build a simple MONDO ID -> disease name lookup."""
    return pd.DataFrame([{
        'mondo_id':       row['formatted_id'],
        'disease_name':   row['Preferred Label'],
        'search_keywords': row['Preferred Label'].lower().split(),
    } for _, row in mondo_df.iterrows()])


def match_mondo_to_hpo(mondo_mapping, hpo_df):
    """Match MONDO diseases to HPO records by disease name keywords."""
    print("\nMatching MONDO to HPO...")
    stop_words = {'disease', 'syndrome', 'disorder', 'type'}
    matched_records = []

    for _, mondo_row in mondo_mapping.iterrows():
        mondo_id = mondo_row['mondo_id']
        disease_name = mondo_row['disease_name'].lower()
        keywords = [w for w in disease_name.split() if len(w) > 4 and w not in stop_words]
        if not keywords:
            keywords = disease_name.split()[:3]

        hpo_names = hpo_df['disease_name'].fillna('').str.lower()
        mask = pd.Series(False, index=hpo_df.index)
        for kw in keywords:
            try:
                mask |= hpo_names.str.contains(kw, case=False, na=False, regex=False)
            except Exception:
                continue

        matched = hpo_df[mask].copy()
        if len(matched) > 0:
            matched['mondo_id'] = mondo_id
            matched['mondo_disease_name'] = mondo_row['disease_name']
            matched_records.append(matched)
            db_ids = ', '.join(matched['database_id'].unique()[:3])
            print(f"  {mondo_id}: {len(matched)} records (DB: {db_ids})")
        else:
            print(f"  {mondo_id}: no match")

    if matched_records:
        result = pd.concat(matched_records, ignore_index=True)
        print(f"\nMatched {len(result)} records across {result['mondo_id'].nunique()} diseases")
        return result
    print("  No matches found.")
    return pd.DataFrame()


# ── Feature extraction ───────────────────────────────────────────────────────

def extract_organ_system_features(matched_df):
    """Build organ system one-hot matrix using vectorized keyword search."""
    print("\nExtracting organ system features (vectorized)...")
    mondo_ids = matched_df['mondo_id'].unique()
    mondo_to_idx = {mid: i for i, mid in enumerate(mondo_ids)}
    organ_names = list(HPO_ORGAN_SYSTEMS.keys())
    organ_matrix = np.zeros((len(mondo_ids), len(organ_names)), dtype=np.int8)

    # Pre-build search text once
    matched_df = matched_df.copy()
    matched_df['search_text'] = (
        matched_df['disease_name'].fillna('') + ' ' + matched_df['hpo_id'].fillna('')
    ).str.lower()

    for sys_idx, (system_name, system_info) in enumerate(HPO_ORGAN_SYSTEMS.items()):
        print(f"  [{sys_idx+1}/{len(HPO_ORGAN_SYSTEMS)}] {system_name}", end='\r')
        mask = pd.Series(False, index=matched_df.index)
        for kw in system_info['keywords']:
            mask |= matched_df['search_text'].str.contains(kw, na=False, regex=False)
        for mid in matched_df.loc[mask, 'mondo_id'].unique():
            organ_matrix[mondo_to_idx[mid], sys_idx] = 1

    print()
    organ_df = pd.DataFrame(organ_matrix, columns=[f'organ_{n}' for n in organ_names])
    organ_df.insert(0, 'mondo_id', mondo_ids)

    print("\nOrgan system involvement:")
    for col, cnt in organ_df.iloc[:, 1:].sum().items():
        print(f"  {col}: {cnt}/{len(mondo_ids)} ({cnt/len(mondo_ids)*100:.1f}%)")
    return organ_df


def extract_phenotype_statistics(matched_df):
    """Compute per-disease phenotype annotation statistics."""
    print("\nComputing phenotype statistics...")
    stats = matched_df.groupby('mondo_id').agg(
        total_phenotypes    =('hpo_id',    'nunique'),
        total_annotations   =('hpo_id',    'count'),
        frequency_available =('frequency', lambda x: x.notna().mean()),
        literature_support  =('reference', lambda x: x.str.contains('PMID', na=False).sum()),
    ).reset_index()
    disease_names = matched_df[['mondo_id', 'mondo_disease_name']].drop_duplicates()
    stats = stats.merge(disease_names, on='mondo_id', how='left')
    stats['phenotype_richness_score'] = np.log1p(stats['total_phenotypes']) * np.log1p(stats['literature_support'])
    print(f"  Shape: {stats.shape}")
    return stats


def merge_features(organ_df, stats_df):
    """Merge organ and statistics features into a single DataFrame."""
    combined = organ_df.merge(stats_df, on='mondo_id', how='outer')
    id_cols    = ['mondo_id', 'mondo_disease_name']
    organ_cols = [c for c in combined.columns if c.startswith('organ_')]
    stat_cols  = [c for c in combined.columns if c not in id_cols and c not in organ_cols]
    return combined[id_cols + organ_cols + stat_cols]


def analyze_multi_organ_involvement(organ_df):
    """Print distribution of organ system involvement counts."""
    organ_cols = [c for c in organ_df.columns if c.startswith('organ_')]
    counts = organ_df[organ_cols].sum(axis=1)
    print("\nOrgan system count distribution:")
    for i in range(int(counts.max()) + 1):
        n = (counts == i).sum()
        if n:
            print(f"  {i} systems: {n} diseases")

    # Identify systemic diseases (>=4 organ systems)
    systemic = organ_df[counts >= 4]
    if len(systemic):
        print("\nSystemic diseases (>=4 organ systems):")
        for _, row in systemic.iterrows():
            involved = [c.replace('organ_', '') for c in organ_cols if row[c]]
            print(f"  {row['mondo_id']}: {', '.join(involved)}")


# ── Main ─────────────────────────────────────────────────────────────────────

def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    test_mode = '--test' in sys.argv
    if test_mode:
        print("[Test mode] Processing first 3 diseases only.")

    mondo_df = load_mondo_list(MONDO_FILE)
    if test_mode:
        mondo_df = mondo_df.head(3)

    hpo_df = load_hpo_annotations(HPO_FILE)
    if hpo_df is None:
        return

    print_db_sources(hpo_df)

    mondo_mapping = build_mondo_mapping(mondo_df)
    matched_df    = match_mondo_to_hpo(mondo_mapping, hpo_df)

    if matched_df.empty:
        print("No matching data found.")
        return

    # Save MONDO <-> HPO mapping for inspection
    mapping_info = matched_df[['mondo_id', 'mondo_disease_name', 'database_id', 'disease_name']].drop_duplicates()
    mapping_info.to_csv(OUTPUT_MAPPING_FILE, index=False)
    print(f"\nMapping saved to: {OUTPUT_MAPPING_FILE}")

    organ_features   = extract_organ_system_features(matched_df)
    phenotype_stats  = extract_phenotype_statistics(matched_df)
    combined_features = merge_features(organ_features, phenotype_stats)

    organ_features.to_csv(OUTPUT_ORGAN_FILE, index=False)
    phenotype_stats.to_csv(OUTPUT_PHENOTYPE_FILE, index=False)
    print(f"\nOrgan features saved to:    {OUTPUT_ORGAN_FILE}")
    print(f"Phenotype stats saved to:   {OUTPUT_PHENOTYPE_FILE}")

    analyze_multi_organ_involvement(organ_features)

    print(f"\nSummary:")
    print(f"  Organ system features: {organ_features.shape}")
    print(f"  Phenotype statistics:  {phenotype_stats.shape}")
    print(f"  Organ dims: {len(HPO_ORGAN_SYSTEMS)} | Stat dims: 5 | Total: {len(HPO_ORGAN_SYSTEMS) + 5}")
    print(combined_features.head())


if __name__ == '__main__':
    main()